In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from collections import defaultdict
from datetime import datetime  # 🔥 para timestamp

base_url = "https://www.football-espana.net"
url = base_url + "/category/la-liga/real-madrid"

headers = {
    "User-Agent": "Mozilla/5.0"
}

# 🔥 FUNCIÓN UNIVERSAL + AUTOR
def get_article_universal_recursivo(url):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        raw_data = defaultdict(list)

        # 🔹 URL
        raw_data["url"] = url

        # 🔹 EXTRAER TODO POR TAGS (BRONZE)
        for tag_name in ['h1', 'h2', 'h3', 'h4', 'h5', 'p', 'span', 'div', 'a', 'li', 'strong', 'em', 'time']:
            elements = soup.find_all(tag_name)
            texts = [el.get_text(strip=True) for el in elements if el.get_text(strip=True)]
            raw_data[f'{tag_name}_all_texts'] = texts[:10]

        # 🔹 ATRIBUTOS
        raw_data['all_hrefs'] = [a.get('href', '') for a in soup.find_all('a', href=True)][:20]
        raw_data['all_src'] = [img.get('src', '') for img in soup.find_all('img')][:10]

        raw_data['all_classes'] = list(set(
            cls for el in soup.find_all() for cls in el.get('class', []) if cls
        ))

        # 🔹 META TAGS
        metas = {}
        for meta in soup.find_all('meta'):
            key = meta.get('property') or meta.get('name')
            value = meta.get('content')
            if key and value:
                metas[key] = value

        raw_data['all_meta'] = metas

        # 🔹 TIME TAGS
        time_tags = soup.find_all("time")
        time_data = []
        for t in time_tags:
            time_data.append({
                "text": t.get_text(strip=True),
                "datetime": t.get("datetime"),
                "class": t.get("class", [])
            })

        raw_data["time_all"] = time_data

        # 🔥 AUTOR Y FECHA
        author = None
        date = None

        for block in soup.find_all(["p", "span", "div"]):
            text = block.get_text("\n", strip=True)

            if "Posted by" in text:
                lines = [l.strip() for l in text.split("\n") if l.strip()]

                if "Posted by" in lines:
                    idx = lines.index("Posted by")

                    if idx + 1 < len(lines):
                        author = lines[idx + 1]

                    if idx + 2 < len(lines):
                        date = lines[idx + 2]

                break

        raw_data["author"] = author
        raw_data["published_time"] = date

        # 🔹 fallback desde meta
        if not raw_data["published_time"]:
            raw_data["published_time"] = (
                metas.get("article:published_time") or
                metas.get("published_time") or
                metas.get("pubdate")
            )

        return dict(raw_data)

    except Exception as e:
        return {
            'url': url,
            'error': str(e),
            'status': 'error'
        }


# 🔹 Obtener página principal
response = requests.get(url, headers=headers, timeout=10)
soup = BeautifulSoup(response.text, "html.parser")

posts = soup.find_all("div", class_="post-container")

data = []

for post in posts:
    link_tag = post.find("a", href=True)
    link = link_tag.get("href") if link_tag else None

    if link and link.startswith("/"):
        link = base_url + link

    if link:
        print(f"🔄 Procesando: {link}")
        article_raw = get_article_universal_recursivo(link)
        data.append(article_raw)


# 📊 DataFrame
df_raw_universal = pd.DataFrame(data)

# 🔥 GENERAR NOMBRE CON TIMESTAMP
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"football_espana_real_madrid_{timestamp}.csv"

# 💾 Guardar
df_raw_universal.to_csv(filename, index=False)

print(f"\n📁 Archivo guardado: {filename}")
print(f"🚀 TOTAL ARTÍCULOS: {len(df_raw_universal)}")

🔄 Procesando: https://www.football-espana.net/2026/03/22/juventus-offer-real-madrid-defender
🔄 Procesando: https://www.football-espana.net/2026/03/22/real-madrid-angry-international-call-ups
🔄 Procesando: https://www.football-espana.net/2026/03/21/rudiger-arbeloa-example
🔄 Procesando: https://www.football-espana.net/2026/03/21/alvaro-arbeloa-jude-bellingham-real-madrid
🔄 Procesando: https://www.football-espana.net/2026/03/21/simeone-arbeloa-bernabeu
🔄 Procesando: https://www.football-espana.net/2026/03/21/real-madrid-defender-injury-return-date
🔄 Procesando: https://www.football-espana.net/2026/03/21/man-united-target-real-madrid
🔄 Procesando: https://www.football-espana.net/2026/03/20/the-good-the-bad-the-beautiful-the-voodoo-child-end-of-an-era-and-initial-reaction-to-genius
🔄 Procesando: https://www.football-espana.net/2026/03/20/andoni-iraola-favourite-to-take-over-la-liga-giants-this-summer
🔄 Procesando: https://www.football-espana.net/2026/03/20/alvaro-arbeloa-v-diego-simeone-the

In [7]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
df_raw_universal.info()
df_raw_universal.head()
 # 50 chars preview

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   url               20 non-null     object
 1   h1_all_texts      20 non-null     object
 2   h2_all_texts      20 non-null     object
 3   h3_all_texts      20 non-null     object
 4   h4_all_texts      20 non-null     object
 5   h5_all_texts      20 non-null     object
 6   p_all_texts       20 non-null     object
 7   span_all_texts    20 non-null     object
 8   div_all_texts     20 non-null     object
 9   a_all_texts       20 non-null     object
 10  li_all_texts      20 non-null     object
 11  strong_all_texts  20 non-null     object
 12  em_all_texts      20 non-null     object
 13  time_all_texts    20 non-null     object
 14  all_hrefs         20 non-null     object
 15  all_src           20 non-null     object
 16  all_classes       20 non-null     object
 17  all_meta          

,url,h1_all_texts,h2_all_texts,h3_all_texts,h4_all_texts,h5_all_texts,p_all_texts,span_all_texts,div_all_texts,a_all_texts,li_all_texts,strong_all_texts,em_all_texts,time_all_texts,all_hrefs,all_src,all_classes,all_meta,time_all,author,published_time
0,https://www.football-espana.net/2026/03/22/juv...,[Juventus readying offer for Real Madrid defen...,[Rudiger exit would be big blow for Alvaro Arb...,"[Live Comments, Leave a ReplyCancel reply]",[],[],"[Real Madrid will be busy this summer, and amo...","[Luciano Spalletti, whom he worked under at Ro...",[HomeLatest NewsAnalysisLa LigaAlavesAthletic ...,"[Home, Latest News, Analysis, La Liga, Alaves,...","[Home, Latest News, Analysis, La LigaAlavesAth...",[],[],"[22 March 2026, 3:12, 22 March 2026, 3:12, 22 ...","[https://www.football-espana.net/, /, /categor...",[https://icdn.football-espana.net/wp-content/u...,"[a2a_dd, form-submit, std-pad, icon, boxout, s...","{'viewport': 'width=device-width, initial-scal...","[{'text': '22 March 2026, 3:12', 'datetime': '...",John Menzies,"22 March 2026, 3:12"
1,https://www.football-espana.net/2026/03/22/rea...,[Real Madrid “nervous and angry” after two pla...,[Real Madrid particularly dumbfounded at Belli...,"[Live Comments, Leave a ReplyCancel reply]",[],[],[Real Madrid host Atletico Madrid on Sunday be...,"[Tags, Add a Comment, Your email address will ...",[HomeLatest NewsAnalysisLa LigaAlavesAthletic ...,"[Home, Latest News, Analysis, La Liga, Alaves,...","[Home, Latest News, Analysis, La LigaAlavesAth...",[],[],"[22 March 2026, 2:41, 22 March 2026, 2:41, 21 ...","[https://www.football-espana.net/, /, /categor...",[https://icdn.football-espana.net/wp-content/u...,"[a2a_dd, form-submit, std-pad, icon, boxout, s...","{'viewport': 'width=device-width, initial-scal...","[{'text': '22 March 2026, 2:41', 'datetime': '...",John Menzies,"22 March 2026, 2:41"
2,https://www.football-espana.net/2026/03/21/rud...,[‘Antonio Rudiger is an example for young play...,"[‘Rudiger is an example for young players’, ‘H...","[Live Comments, Leave a ReplyCancel reply]",[],[],[Real Madrid manager Alvaro Arbeloa has defend...,"[Tags, Add a Comment, Your email address will ...",[HomeLatest NewsAnalysisLa LigaAlavesAthletic ...,"[Home, Latest News, Analysis, La Liga, Alaves,...","[Home, Latest News, Analysis, La LigaAlavesAth...",[],[],"[21 March 2026, 17:48, 21 March 2026, 17:48, 2...","[https://www.football-espana.net/, /, /categor...",[https://icdn.football-espana.net/wp-content/u...,"[a2a_dd, form-submit, std-pad, icon, boxout, s...","{'viewport': 'width=device-width, initial-scal...","[{'text': '21 March 2026, 17:48', 'datetime': ...",Ruairidh Barlow,"21 March 2026, 17:48"
3,https://www.football-espana.net/2026/03/21/alv...,[Alvaro Arbeloa explains how Jude Bellingham w...,[Arbeloa: Bellingham return won’t affect Thiag...,"[1 Comment, Leave a ReplyCancel reply]",[],[],[Real Madrid are back in La Liga action on Sun...,[“Tomorrow is a very important match for us be...,[HomeLatest NewsAnalysisLa LigaAlavesAthletic ...,"[Home, Latest News, Analysis, La Liga, Alaves,...","[Home, Latest News, Analysis, La LigaAlavesAth...",[],[],"[21 March 2026, 17:03, 21 March 2026, 17:03, 2...","[https://www.football-espana.net/, /, /categor...",[https://icdn.football-espana.net/wp-content/u...,"[a2a_dd, form-submit, std-pad, icon, boxout, c...","{'viewport': 'width=device-width, initial-scal...","[{'text': '21 March 2026, 17:03', 'datetime': ...",John Menzies,"21 March 2026, 17:03"
4,https://www.football-espana.net/2026/03/21/sim...,[Diego Simeone praises Alvaro Arbeloa before M...,"[Changes to Real Madrid since first meeting, W...",[‘Everyone has their ups and downs’ – Simeone ...,[],[],[Atletico Madrid manager Diego Simeone has bee...,"[Tags, Add a Comment, Your email address will ...",[HomeLatest NewsAnalysisLa LigaAlavesAthletic ...,"[Home, Latest News, Analysis, La Liga, Alaves,...","[Home, Latest News, Analysis, La LigaAlavesAth...",[],[],"[21 March 2026, 15:35, 21 March 2